In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

In [6]:
np.random.seed(42)

# --- 1. Synthesize Customers & Orders ---
# Olist typically has ~99k orders. Let's do 50k unique customers, some with repeat purchases.
n_unique_customers = 50000

# 95% one-time, 4% two-time, 1% 3+ times
purchase_counts = np.random.choice([1, 2, 3, 4, 5], p=[0.95, 0.04, 0.007, 0.002, 0.001], size=n_unique_customers)
total_orders = purchase_counts.sum()

unique_customer_ids = [f"c_uniq_{i}" for i in range(n_unique_customers)]
customer_unique_id_col = np.repeat(unique_customer_ids, purchase_counts)
order_ids = [f"ord_{i}" for i in range(total_orders)]
customer_ids = [f"c_ord_{i}" for i in range(total_orders)] # 1-to-1 with orders

# Create timestamps
start_date = pd.to_datetime('2017-01-01')
# For customers with multiple orders, ensure chronological order
order_purchase_timestamp = []
for counts in purchase_counts:
    base_date = start_date + pd.to_timedelta(np.random.randint(0, 500), unit='D')
    dates = [base_date]
    for _ in range(1, counts):
        # Time to next purchase (lognormal-ish, median ~40 days)
        next_date = dates[-1] + pd.to_timedelta(np.random.randint(5, 120), unit='D')
        dates.append(next_date)
    order_purchase_timestamp.extend(dates)

df_orders = pd.DataFrame({
    'order_id': order_ids,
    'customer_id': customer_ids,
    'order_purchase_timestamp': order_purchase_timestamp,
    'order_status': 'delivered'
})

df_customers = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_unique_id': customer_unique_id_col
})

# --- 2. Synthesize Payments (GMV) ---
# Log-normal distribution for order value
order_values = np.random.lognormal(mean=4.5, sigma=0.8, size=total_orders)
df_payments = pd.DataFrame({
    'order_id': order_ids,
    'payment_value': order_values
})

# --- 3. Synthesize Reviews & Delivery Experience ---
# Simulated Delivery delays
delivery_delays = np.random.normal(-5, 4, size=total_orders) # Negative means early
df_orders['estimated_delivery_date'] = df_orders['order_purchase_timestamp'] + pd.to_timedelta(15, unit='D')
df_orders['delivered_customer_date'] = df_orders['estimated_delivery_date'] + pd.to_timedelta(delivery_delays, unit='D')

# Reviews: dependent on delay
review_scores = []
for delay in delivery_delays:
    if delay > 0: # Late
        review_scores.append(np.random.choice([1, 2, 3, 4, 5], p=[0.6, 0.15, 0.1, 0.1, 0.05]))
    else: # On time / Early
        review_scores.append(np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.05, 0.1, 0.2, 0.6]))
        
df_reviews = pd.DataFrame({
    'order_id': order_ids,
    'review_score': review_scores
})

print("Synthetic Dataset Created:")
print(f"Total Orders: {total_orders}")
print(f"Total Unique Customers: {n_unique_customers}")

Synthetic Dataset Created:
Total Orders: 53171
Total Unique Customers: 50000


In [7]:
# Merge the core dataset
df = df_orders.merge(df_customers, on='customer_id')\
              .merge(df_payments, on='order_id')\
              .merge(df_reviews, on='order_id')

# Ensure datetime
for col in ['order_purchase_timestamp', 'estimated_delivery_date', 'delivered_customer_date']:
    df[col] = pd.to_datetime(df[col])

df['late_delivery'] = (df['delivered_customer_date'] > df['estimated_delivery_date']).astype(int)

# --- 1. Customer Identifiers ---
unique_customer_ids = df['customer_id'].nunique()
unique_person_ids = df['customer_unique_id'].nunique()
mult_order_persons = (df.groupby('customer_unique_id').size() > 1).sum()

print(f"Unique customer_id (Orders): {unique_customer_ids}")
print(f"Unique customer_unique_id (People): {unique_person_ids}")
print(f"People with >1 order: {mult_order_persons}")
print(f"Overstatement if using customer_id: {(unique_customer_ids / unique_person_ids - 1) * 100:.1f}%\n")

# --- 2 & 3. Customer Types & Composition ---
customer_stats = df.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('payment_value', 'sum'),
    first_order_date=('order_purchase_timestamp', 'min'),
    avg_review=('review_score', 'mean')
).reset_index()

customer_stats['customer_type'] = np.where(customer_stats['total_orders'] == 1, 'One-time', 'Repeat')

type_counts = customer_stats['customer_type'].value_counts()
print("--- Customer Base Composition ---")
print(type_counts)
print(f"Repeat Customer Rate: {type_counts['Repeat'] / len(customer_stats) * 100:.2f}%\n")

# Chart 1: Order Frequency Distribution
order_freq = customer_stats['total_orders'].value_counts().sort_index()
plt.figure(figsize=(8, 4))
sns.barplot(x=order_freq.index, y=order_freq.values, color='steelblue')
plt.title('Distribution of Orders per Customer')
plt.xlabel('Number of Orders')
plt.ylabel('Number of Customers')
plt.yscale('log') # Log scale to make repeat customers visible
plt.close()

# --- 4 & 5. Revenue & Economic Value ---
revenue_comparison = customer_stats.groupby('customer_type').agg(
    customer_count=('customer_unique_id', 'count'),
    total_gmv=('total_revenue', 'sum'),
    mean_lifetime_revenue=('total_revenue', 'mean'),
    median_lifetime_revenue=('total_revenue', 'median'),
    mean_orders=('total_orders', 'mean')
).reset_index()

revenue_comparison['gmv_share_pct'] = (revenue_comparison['total_gmv'] / revenue_comparison['total_gmv'].sum()) * 100
revenue_comparison['customer_share_pct'] = (revenue_comparison['customer_count'] / revenue_comparison['customer_count'].sum()) * 100

print("--- Economic Value Comparison ---")
print(revenue_comparison.to_string())

# Calculate Value Multiplier
avg_rev_one_time = revenue_comparison.loc[revenue_comparison['customer_type'] == 'One-time', 'mean_lifetime_revenue'].values[0]
avg_rev_repeat = revenue_comparison.loc[revenue_comparison['customer_type'] == 'Repeat', 'mean_lifetime_revenue'].values[0]
print(f"\nA Repeat Customer is {avg_rev_repeat / avg_rev_one_time:.2f}x more valuable on average.\n")

# Chart 2: Customer vs GMV Share
plt.figure(figsize=(8, 4))
x = np.arange(2)
width = 0.35
plt.bar(x - width/2, revenue_comparison['customer_share_pct'], width, label='% of Customers', color='lightgray')
plt.bar(x + width/2, revenue_comparison['gmv_share_pct'], width, label='% of GMV', color='darkorange')
plt.xticks(x, revenue_comparison['customer_type'])
plt.ylabel('Percentage (%)')
plt.title('Customer Count Share vs. GMV Share')
plt.legend()
plt.close()

Unique customer_id (Orders): 53171
Unique customer_unique_id (People): 50000
People with >1 order: 2470
Overstatement if using customer_id: 6.3%

--- Customer Base Composition ---
customer_type
One-time    47530
Repeat       2470
Name: count, dtype: int64
Repeat Customer Rate: 4.94%

--- Economic Value Comparison ---
  customer_type  customer_count     total_gmv  mean_lifetime_revenue  median_lifetime_revenue  mean_orders  gmv_share_pct  customer_share_pct
0      One-time           47530  5.887981e+06             123.879265                89.904593     1.000000      89.534174               95.06
1        Repeat            2470  6.882578e+05             278.646872               223.139244     2.283806      10.465826                4.94

A Repeat Customer is 2.25x more valuable on average.

